
# Benchmark por Planta — Estrategias y Métricas de Drift

Este notebook arma **resúmenes por planta** leyendo los archivos exportados en `reportes_Drift*/plantaX/`:
- `plantaX_decay_metrics.csv`, `plantaX_golden_metrics.csv`, `plantaX_seasonal_metrics.csv`
- `/_comparisons/plantaX_summary_all_metrics.csv` y `/_comparisons/plantaX_columns_all_metrics.csv`

Produce:
- Heatmaps de **drift rate** por estrategia y métrica
- **Correlación** entre métricas por columna
- **Top columnas** más sensibles por métrica
- Archivos PNG por planta en la carpeta `_benchmarks/`


## 1) Setup

In [5]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ajusta ROOT a tu carpeta de reportes (por ejemplo 'reportes_DriftMetrics' o 'reportes_Drift')
ROOT = Path('reportes_DriftMetricas')
PLANTS = ['planta1','planta2','planta3']
OUTDIR = ROOT / '_benchmarks'
OUTDIR.mkdir(parents=True, exist_ok=True)

def safe_read_csv(p):
    try:
        return pd.read_csv(p)
    except Exception as e:
        print(f"[WARN] No pude leer {p}: {e}")
        return pd.DataFrame()


## 2) Funciones auxiliares

In [6]:

def plot_drift_rate_heatmap(summary_df, plant, save=True):
    sub = summary_df.query("plant == @plant").copy()
    if sub.empty:
        print(f"[{plant}] Sin resumen para heatmap")
        return None
    pv = sub.pivot_table(index='strategy', columns='metric', values='drift_rate_pct', aggfunc='mean')
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,4))
    im = plt.imshow(pv.values, aspect='auto')
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.yticks(range(pv.shape[0]), pv.index)
    plt.xticks(range(pv.shape[1]), pv.columns, rotation=30, ha='right')
    plt.title(f"Drift rate (%) — {plant}")
    plt.tight_layout()
    if save:
        path = OUTDIR / f"{plant}_heatmap_drift_rate.png"
        plt.savefig(path, dpi=160)
        plt.close()
        return path
    else:
        plt.show()
        return None

def corr_between_metrics(columns_df, plant, strategy):
    g = columns_df.query("plant == @plant and strategy == @strategy").copy()
    if g.empty:
        return None, None
    pv = g.pivot_table(index='col', columns='metric', values='score', aggfunc='mean')
    pv = pv.dropna(how='all')
    if pv.shape[1] < 2 or pv.dropna().shape[0] < 3:
        return pv, None
    cor = pv.corr()
    return pv, cor

def top_sensitive(columns_df, plant, metric, strategy=None, k=10):
    df = columns_df.query("plant == @plant and metric == @metric").copy()
    if strategy is not None:
        df = df.query("strategy == @strategy")
    if df.empty:
        return pd.DataFrame()
    out = (df[['col','strategy','score']]
           .groupby(['col','strategy'], as_index=False)['score'].mean()
           .sort_values('score', ascending=False)
           .head(k))
    return out


## 3) Carga global (consolidados por planta)

In [7]:

sum_list, col_list = [], []
for plant in PLANTS:
    s = ROOT / plant / "_comparisons" / f"{plant}_summary_all_metrics.csv"
    c = ROOT / plant / "_comparisons" / f"{plant}_columns_all_metrics.csv"
    sdf = safe_read_csv(s); cdf = safe_read_csv(c)
    if not sdf.empty:
        sum_list.append(sdf)
    if not cdf.empty:
        col_list.append(cdf)

df_sum_all = pd.concat(sum_list, ignore_index=True) if sum_list else pd.DataFrame()
df_cols_all = pd.concat(col_list, ignore_index=True) if col_list else pd.DataFrame()

display(df_sum_all.head())
display(df_cols_all.head())


,ref_rows,cur_rows,n_columns_total,n_columns_drifted,drift_rate_pct,n_numeric,n_categorical,plant,strategy,metric
0,31666,3580,15,0,0.000000,14,1,planta1,decay,evidently_default
1,31666,3580,15,10,66.666667,14,1,planta1,decay,ks
2,31666,3580,15,6,40.000000,14,1,planta1,decay,mannwhitney
3,31666,3580,15,11,73.333333,14,1,planta1,decay,psi
4,31666,3580,15,7,46.666667,14,1,planta1,decay,wasserstein


,col,type,score,threshold,method,drift_detected,ref_count,cur_count,ref_missing_pct,cur_missing_pct,ref_mean,cur_mean,plant,strategy,metric
0,Conductividad DAF,numeric,0.332936,0.15,ks,True,NaN,NaN,NaN,NaN,NaN,NaN,planta1,decay,ks
1,Flujo Parshall 01 entrada a Ecualizador 1,numeric,0.110098,0.15,ks,False,NaN,NaN,NaN,NaN,NaN,NaN,planta1,decay,ks
2,Flujo de envío a DAF,numeric,0.113706,0.15,ks,False,NaN,NaN,NaN,NaN,NaN,NaN,planta1,decay,ks
3,Nivel Clarificado DAF (Tk 80m3),numeric,0.408967,0.15,ks,True,NaN,NaN,NaN,NaN,NaN,NaN,planta1,decay,ks
4,Nivel Ecualizador 1 (Tk 30m3),numeric,0.598792,0.15,ks,True,NaN,NaN,NaN,NaN,NaN,NaN,planta1,decay,ks


## 4) Reporte por planta

In [8]:

reports = []
for plant in PLANTS:
    heatmap_path = plot_drift_rate_heatmap(df_sum_all, plant, save=True)

    corrs = []
    for strat in ['decay','golden','seasonal']:
        pv, cor = corr_between_metrics(df_cols_all, plant, strat)
        if cor is not None:
            import numpy as np
            mean_corr = float(cor.where(~np.eye(len(cor), dtype=bool)).stack().mean())
        else:
            mean_corr = np.nan
        corrs.append({'plant': plant, 'strategy': strat, 'corr_mean': mean_corr})
        if cor is not None:
            import matplotlib.pyplot as plt
            plt.figure(figsize=(4,3))
            im = plt.imshow(cor.values, vmin=-1, vmax=1, aspect='equal')
            plt.colorbar(im, fraction=0.046, pad=0.04)
            plt.xticks(range(cor.shape[1]), cor.columns, rotation=30, ha='right')
            plt.yticks(range(cor.shape[0]), cor.index)
            plt.title(f"Corr métricas — {plant} · {strat}")
            plt.tight_layout()
            plt.savefig(OUTDIR / f"{plant}_{strat}_corr_metrics.png", dpi=160)
            plt.close()

    corr_df = pd.DataFrame(corrs)
    tops = []
    for metric in ['ks','mannwhitney','psi','wasserstein']:
        t = top_sensitive(df_cols_all, plant, metric, strategy=None, k=8)
        t['metric'] = metric
        tops.append(t)
    tops_df = pd.concat(tops, ignore_index=True) if tops else pd.DataFrame()

    if not corr_df.empty:
        corr_df.to_csv(OUTDIR / f"{plant}_corr_summary.csv", index=False)
    if not tops_df.empty:
        tops_df.to_csv(OUTDIR / f"{plant}_top_columns_by_metric.csv", index=False)

    reports.append({'plant': plant, 'heatmap': str(heatmap_path) if heatmap_path else None})

import pandas as pd
pd.DataFrame(reports)


,plant,heatmap
0,planta1,reportes_DriftMetricas\_benchmarks\planta1_hea...
1,planta2,reportes_DriftMetricas\_benchmarks\planta2_hea...
2,planta3,reportes_DriftMetricas\_benchmarks\planta3_hea...



## 5) Guía para el informe

- **Sensibilidad**: usa el heatmap por planta para comparar combinaciones estrategia–métrica.
- **Concordancia**: `*_corr_summary.csv` reporta `corr_mean`; si es baja, revisar bins/umbrales.
- **Variables clave**: `*_top_columns_by_metric.csv` lista columnas con mayor `score` promedio (buenas para ejemplos).


In [9]:
df_sum_all

,ref_rows,cur_rows,n_columns_total,n_columns_drifted,drift_rate_pct,n_numeric,n_categorical,plant,strategy,metric
0,31666,3580,15,0,0.000000,14,1,planta1,decay,evidently_default
1,31666,3580,15,10,66.666667,14,1,planta1,decay,ks
2,31666,3580,15,6,40.000000,14,1,planta1,decay,mannwhitney
3,31666,3580,15,11,73.333333,14,1,planta1,decay,psi
4,31666,3580,15,7,46.666667,14,1,planta1,decay,wasserstein
5,1104,3580,13,0,0.000000,12,1,planta1,golden,evidently_default
6,1104,3580,13,13,100.000000,12,1,planta1,golden,ks
7,1104,3580,13,6,46.153846,12,1,planta1,golden,mannwhitney
8,1104,3580,13,12,92.307692,12,1,planta1,golden,psi
9,1104,3580,13,12,92.307692,12,1,planta1,golden,wasserstein


In [10]:
df_cols_all

,col,type,score,threshold,method,drift_detected,ref_count,cur_count,ref_missing_pct,cur_missing_pct,ref_mean,cur_mean,plant,strategy,metric
0,Conductividad DAF,numeric,0.332936,0.150000,ks,True,NaN,NaN,NaN,NaN,NaN,NaN,planta1,decay,ks
1,Flujo Parshall 01 entrada a Ecualizador 1,numeric,0.110098,0.150000,ks,False,NaN,NaN,NaN,NaN,NaN,NaN,planta1,decay,ks
2,Flujo de envío a DAF,numeric,0.113706,0.150000,ks,False,NaN,NaN,NaN,NaN,NaN,NaN,planta1,decay,ks
3,Nivel Clarificado DAF (Tk 80m3),numeric,0.408967,0.150000,ks,True,NaN,NaN,NaN,NaN,NaN,NaN,planta1,decay,ks
4,Nivel Ecualizador 1 (Tk 30m3),numeric,0.598792,0.150000,ks,True,NaN,NaN,NaN,NaN,NaN,NaN,planta1,decay,ks
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339,Nivel pozo,numeric,0.398866,0.200000,psi,True,NaN,NaN,NaN,NaN,NaN,NaN,planta3,seasonal,psi
340,Presión pozo,numeric,inf,0.200000,psi,True,NaN,NaN,NaN,NaN,NaN,NaN,planta3,seasonal,psi
341,Energía Total Activa Bombas Elevadoras,numeric,3.708210,0.753145,wasserstein,True,NaN,NaN,NaN,NaN,NaN,NaN,planta3,seasonal,wasserstein
342,Nivel pozo,numeric,0.266073,0.017573,wasserstein,True,NaN,NaN,NaN,NaN,NaN,NaN,planta3,seasonal,wasserstein


In [11]:
reports

[{'plant': 'planta1',
  'heatmap': 'reportes_DriftMetricas\\_benchmarks\\planta1_heatmap_drift_rate.png'},
 {'plant': 'planta2',
  'heatmap': 'reportes_DriftMetricas\\_benchmarks\\planta2_heatmap_drift_rate.png'},
 {'plant': 'planta3',
  'heatmap': 'reportes_DriftMetricas\\_benchmarks\\planta3_heatmap_drift_rate.png'}]